In [ ]:
# Clone repository and setup
!git clone https://github.com/dedeepyakm/NS-MCA-Clinical-AI.git
%cd NS-MCA-Clinical-AI

# Verify we're in right directory
!pwd
!ls -la

fatal: destination path 'NS-MCA-Clinical-AI' already exists and is not an empty directory.
/content/NS-MCA-Clinical-AI
/content/NS-MCA-Clinical-AI
total 44
drwxr-xr-x 6 root root 4096 May 16 03:15 .
drwxr-xr-x 1 root root 4096 May 16 03:15 ..
-rw-r--r-- 1 root root 1763 May 16 03:15 ARCHITECTURE.md
drwxr-xr-x 2 root root 4096 May 16 03:15 config
drwxr-xr-x 2 root root 4096 May 16 03:15 data
-rw-r--r-- 1 root root  376 May 16 03:15 .env
drwxr-xr-x 8 root root 4096 May 16 03:15 .git
-rw-r--r-- 1 root root  487 May 16 03:15 .gitignore
-rw-r--r-- 1 root root 1067 May 16 03:15 LICENSE
-rw-r--r-- 1 root root  304 May 16 03:15 requirements.txt
drwxr-xr-x 2 root root 4096 May 16 03:15 results


In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
!mkdir -p '/content/drive/My Drive/NS-MCA-Results'
!mkdir -p '/content/NS-MCA-Clinical-AI/data'
!mkdir -p '/content/NS-MCA-Clinical-AI/results'

print("✓ Google Drive mounted")
print("✓ Output directories created")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted
✓ Output directories created


In [ ]:
# Install all dependencies
!pip install -q torch transformers numpy pandas scikit-learn scipy pyyaml jsonlines matplotlib seaborn tqdm python-dotenv jupyter ipython ipywidgets

# Install spaCy with pre-built wheels
!pip install -q spacy

# Download spaCy model
!python -m spacy download en_core_web_sm

# Install ScispaCy
!pip install -q scispacy
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_sm-0.5.1.tar.gz

print("✓ All dependencies installed in Colab")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
✓ All depend

In [ ]:
# Verify core packages (skip scispacy for now- becuase of error above, skip pandas for now - has numpy conflict)
import torch
import transformers
import numpy as np
import spacy
import sklearn
import matplotlib
import seaborn
import json
import yaml

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ NumPy: {np.__version__}")
print(f"✓ SpaCy: {spacy.__version__}")
print(f"✓ Scikit-learn: {sklearn.__version__}")
print(f"✓ Matplotlib: {matplotlib.__version__}")
print(f"✓ Seaborn: {seaborn.__version__}")
print(f"✓ JSON: loaded")
print(f"✓ YAML: loaded")

# Test spaCy
nlp = spacy.load("en_core_web_sm")
doc = nlp("The patient took amoxicillin 500mg twice daily")
print(f"\n✓ SpaCy NER test:")
for ent in doc.ents:
    print(f"  {ent.text} → {ent.label_}")

print("\n✓ All core packages verified!")
print("Note: pandas will work in Notebook 00 after runtime restart")

✓ PyTorch: 2.10.0+cpu
✓ Transformers: 5.0.0
✓ NumPy: 1.26.4
✓ SpaCy: 3.8.14
✓ Scikit-learn: 1.6.1
✓ Matplotlib: 3.10.0
✓ Seaborn: 0.13.2
✓ JSON: loaded
✓ YAML: loaded

✓ SpaCy NER test:
  500 → CARDINAL

✓ All core packages verified!
Note: pandas will work in Notebook 00 after runtime restart


In [ ]:
# Load and verify config files
import json
import yaml

# Load YAML config (UTF-8 BOM safe)
with open('config/config.yaml', encoding='utf-8-sig') as f:
    config = yaml.safe_load(f)

# Load JSON configs (UTF-8 BOM safe)
with open('config/entity_types.json', encoding='utf-8-sig') as f:
    entity_types = json.load(f)

with open('config/clinical_policies.json', encoding='utf-8-sig') as f:
    policies = json.load(f)

with open('config/threshold_values.json', encoding='utf-8-sig') as f:
    thresholds = json.load(f)

print("✓ config.yaml loaded")
print("✓ entity_types.json loaded")
print("✓ clinical_policies.json loaded")
print("✓ threshold_values.json loaded")

print(f"\nEntity types: {len(entity_types['entity_types'])}")
print(f"Clinical policies: {len(policies['policies'])}")
print(f"Specialties: {list(thresholds['thresholds'].keys())}")

print("\n✓ All configs loaded successfully!")

✓ config.yaml loaded
✓ entity_types.json loaded
✓ clinical_policies.json loaded
✓ threshold_values.json loaded

Entity types: 7
Clinical policies: 4
Specialties: ['surgery', 'pharmacology', 'general', 'pediatrics']

✓ All configs loaded successfully!


In [ ]:
# Download from public Hugging Face mirror
import json
import os
import urllib.request

print("Downloading MedQA from public source...")

# Create data directory
os.makedirs('data', exist_ok=True)

# Download from public Hugging Face dataset hub (mirror of MedQA)
url = "https://huggingface.co/datasets/AdityaSinghal/MedQA_USMLE/raw/main/data.json"

print("Downloading MedQA-USMLE from Hugging Face...")

try:
    urllib.request.urlretrieve(url, 'data/medqa_raw.json')
    print("✓ Downloaded successfully!")

except Exception as e:
    print(f"Error: {e}")
    print("\nTrying alternative source...")

    # Alternative: Use smaller public sample
    url_alt = "https://raw.githubusercontent.com/AIM-Harvard/MedQA/main/usmle_release/US/qbank.jsonl"
    try:
        urllib.request.urlretrieve(url_alt, 'data/medqa_raw.jsonl')
        print("✓ Downloaded from alternative source!")

        # Convert JSONL to JSON
        import json
        medqa_data = []
        with open('data/medqa_raw.jsonl', 'r') as f:
            for line in f:
                if line.strip():
                    medqa_data.append(json.loads(line))

        with open('data/medqa_raw.json', 'w') as f:
            json.dump(medqa_data, f, indent=2)

        print("✓ Converted to JSON!")

    except Exception as e2:
        print(f"Alternative also failed: {e2}")

# Verify
if os.path.exists('data/medqa_raw.json'):
    with open('data/medqa_raw.json') as f:
        data = json.load(f)
    print(f"\n✓ MedQA dataset loaded!")
    print(f"✓ Total cases: {len(data) if isinstance(data, list) else len(data.get('data', []))}")
else:
    print("\n⚠️ Please manually upload qbank.jsonl")

Error: HTTP Error 401: Unauthorized

Trying alternative source...
Alternative also failed: HTTP Error 404: Not Found

⚠️ Please manually upload qbank.jsonl


In [ ]:
# Find data_clean.zip in Google Drive
import os

# List contents of My Drive
print("Searching for data_clean.zip in Google Drive...\n")

for root, dirs, files in os.walk('/content/drive/My Drive/'):
    for file in files:
        if 'data_clean' in file or file.endswith('.zip'):
            full_path = os.path.join(root, file)
            size_mb = os.path.getsize(full_path) / (1024*1024)
            print(f"✓ Found: {full_path}")
            print(f"  Size: {size_mb:.1f} MB")

            # Save the path for next step
            zip_path = full_path
            break

print("\nIf not found, check:")
print("1. Is data_clean.zip in your Google Drive?")
print("2. Is it in 'My Drive' root or a subfolder?")

Searching for data_clean.zip in Google Drive...

✓ Found: /content/drive/My Drive/data_clean.zip
  Size: 125.6 MB

If not found, check:
1. Is data_clean.zip in your Google Drive?
2. Is it in 'My Drive' root or a subfolder?


In [ ]:
# Copy ZIP from Google Drive to Colab
import shutil
import os

# Path to ZIP in Google Drive (from Step 2)
source_zip = '/content/drive/My Drive/data_clean.zip'

# Destination in Colab
dest_zip = '/content/data_clean.zip'

print(f"Copying from: {source_zip}")
print(f"Copying to: {dest_zip}")

if os.path.exists(source_zip):
    shutil.copy2(source_zip, dest_zip)
    size_mb = os.path.getsize(dest_zip) / (1024*1024)
    print(f"\n✓ File copied successfully!")
    print(f"✓ Size: {size_mb:.1f} MB")
else:
    print("✗ File not found at that location")
    print("Please run previous step again to find the correct path")

Copying from: /content/drive/My Drive/data_clean.zip
Copying to: /content/data_clean.zip

✓ File copied successfully!
✓ Size: 125.6 MB


In [ ]:
# Extract data_clean.zip
import zipfile
import os

zip_file = '/content/data_clean.zip'

print("Extracting data_clean.zip...")
print(f"File size: {os.path.getsize(zip_file) / (1024*1024):.1f} MB")

# Create extraction directory
os.makedirs('/content/data', exist_ok=True)

# Extract
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    total_files = len(zip_ref.namelist())
    print(f"\nTotal files in ZIP: {total_files}")

    # Extract with progress
    print("Extracting...\n")
    for i, file in enumerate(zip_ref.namelist()):
        zip_ref.extract(file, '/content/data/')

        # Progress indicator every 100 files
        if (i + 1) % 100 == 0:
            print(f"  Extracted {i+1}/{total_files} files...")

print(f"\n✓ Extraction complete!")

Extracting data_clean.zip...
File size: 125.6 MB

Total files in ZIP: 310
Extracting...

  Extracted 100/310 files...
  Extracted 200/310 files...
  Extracted 300/310 files...

✓ Extraction complete!


In [ ]:
# Explore extracted contents
import os
import glob
import json

print("=" * 60)
print("EXPLORING EXTRACTED MEDQA DATA")
print("=" * 60)

# Find different file types
jsonl_files = sorted(glob.glob('/content/data/**/*.jsonl', recursive=True))
json_files = sorted(glob.glob('/content/data/**/*.json', recursive=True))
txt_files = sorted(glob.glob('/content/data/**/*.txt', recursive=True))

print(f"\n✓ JSONL files found: {len(jsonl_files)}")
for f in jsonl_files[:10]:
    size = os.path.getsize(f) / (1024*1024)
    filename = os.path.basename(f)
    print(f"  📄 {filename} ({size:.1f} MB)")

print(f"\n✓ JSON files found: {len(json_files)}")
for f in json_files[:5]:
    size = os.path.getsize(f) / (1024*1024)
    filename = os.path.basename(f)
    print(f"  📄 {filename} ({size:.1f} MB)")

print(f"\n✓ TXT files found: {len(txt_files)}")
for f in txt_files[:5]:
    filename = os.path.basename(f)
    print(f"  📄 {filename}")

# Show directory structure
print("\n" + "=" * 60)
print("DIRECTORY STRUCTURE")
print("=" * 60)

for root, dirs, files in os.walk('/content/data/'):
    level = root.replace('/content/data/', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    if level < 3:  # Only show up to 3 levels deep
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:
            print(f"{subindent}{file}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files)-5} more")

EXPLORING EXTRACTED MEDQA DATA

✓ JSONL files found: 30
  📄 dev.jsonl (1.2 MB)
  📄 test.jsonl (1.2 MB)
  📄 train.jsonl (9.7 MB)
  📄 chinese_qbank.jsonl (10.9 MB)
  📄 dev.jsonl (1.3 MB)
  📄 test.jsonl (1.3 MB)
  📄 train.jsonl (10.4 MB)
  📄 dev.jsonl (0.6 MB)
  📄 tw_dev.jsonl (0.9 MB)
  📄 tw_test.jsonl (1.0 MB)

✓ JSON files found: 0

✓ TXT files found: 86
  📄 Anatomy_Gray.txt
  📄 Biochemistry_Lippincott.txt
  📄 Cell_Biology_Alberts.txt
  📄 First_Aid_Step1.txt
  📄 First_Aid_Step2.txt

DIRECTORY STRUCTURE
/
data_clean/
  .DS_Store
  questions/
    .DS_Store
    Taiwan/
      .DS_Store
      taiwanese_qbank.jsonl
      dev.jsonl
      test.jsonl
      train.jsonl
      metamap/
        test/
        train/
        dev/
      tw_translated_jsonl/
        en/
        zh/
    Mainland/
      .DS_Store
      dev.jsonl
      chinese_qbank.jsonl
      test.jsonl
      train.jsonl
      4_options/
    US/
      .DS_Store
      dev.jsonl
      test.jsonl
      US_qbank.jsonl
      train.jsonl
    

In [20]:
# Load US USMLE data only
import json
import os

print("=" * 60)
print("LOADING US USMLE MEDQA DATA")
print("=" * 60)

# Paths to US USMLE data
us_data_path = '/content/data/data_clean/questions/US/'

medqa_data = []

# Load train, dev, test
for split in ['train', 'dev', 'test']:
    jsonl_file = os.path.join(us_data_path, f'{split}.jsonl')

    if os.path.exists(jsonl_file):
        print(f"\nLoading: {split}.jsonl")

        line_count = 0
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        case = json.loads(line)
                        # Add split info for tracking
                        case['split'] = split
                        medqa_data.append(case)
                        line_count += 1
                    except json.JSONDecodeError:
                        pass

        print(f"  ✓ Loaded {line_count} cases")
    else:
        print(f"✗ File not found: {jsonl_file}")

print(f"\n{'='*60}")
print(f"✓ TOTAL US USMLE CASES: {len(medqa_data)}")
print(f"{'='*60}")

# Save unified dataset
output_path = '/content/data/medqa_raw.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(medqa_data, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved to: {output_path}")
print(f"✓ File size: {os.path.getsize(output_path) / (1024*1024):.1f} MB")

# Show structure
if medqa_data:
    print(f"\n{'='*60}")
    print("SAMPLE DATA STRUCTURE")
    print(f"{'='*60}")

    sample = medqa_data[0]
    print(f"\nKeys in each case:")
    for key in sample.keys():
        print(f"  • {key}")

    print(f"\nFirst case (truncated):")
    for key in ['question', 'answer', 'answer_idx', 'options', 'specialty']:
        if key in sample:
            val = str(sample[key])[:70]
            print(f"  {key}: {val}...")

LOADING US USMLE MEDQA DATA

Loading: train.jsonl
  ✓ Loaded 10178 cases

Loading: dev.jsonl
  ✓ Loaded 1272 cases

Loading: test.jsonl
  ✓ Loaded 1273 cases

✓ TOTAL US USMLE CASES: 12723

✓ Saved to: /content/data/medqa_raw.json
✓ File size: 13.4 MB

SAMPLE DATA STRUCTURE

Keys in each case:
  • question
  • answer
  • options
  • meta_info
  • answer_idx
  • split

First case (truncated):
  question: A 23-year-old pregnant woman at 22 weeks gestation presents with burni...
  answer: Nitrofurantoin...
  answer_idx: E...
  options: {'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': 'Ciprofloxacin', 'D': 'Do...


In [19]:
# Analyze US USMLE dataset
import json
from collections import Counter

print("=" * 60)
print("US USMLE MEDQA DATASET ANALYSIS")
print("=" * 60)

with open('/content/data/medqa_raw.json', 'r', encoding='utf-8') as f:
    medqa = json.load(f)

print(f"\n✓ Total US USMLE cases: {len(medqa)}")

if medqa:
    # Split distribution
    print(f"\n✓ Split distribution:")
    splits = Counter([case.get('split', 'unknown') for case in medqa])
    for split, count in sorted(splits.items()):
        percentage = (count / len(medqa)) * 100
        print(f"  {split}: {count} ({percentage:.1f}%)")

    # Specialty distribution
    print(f"\n✓ Specialty distribution:")
    specialties = Counter([case.get('specialty', 'unknown') for case in medqa])
    for spec, count in specialties.most_common():
        percentage = (count / len(medqa)) * 100
        print(f"  {spec}: {count} ({percentage:.1f}%)")

    # Sample questions by specialty
    print(f"\n✓ Sample questions by specialty:")
    shown_specialties = set()
    for case in medqa:
        spec = case.get('specialty', 'unknown')
        if spec not in shown_specialties:
            question = case.get('question', 'N/A')[:60]
            print(f"  [{spec}] {question}...")
            shown_specialties.add(spec)

    # Check data completeness
    print(f"\n✓ Data completeness check:")
    keys_to_check = ['question', 'answer', 'options', 'specialty']
    for key in keys_to_check:
        present = sum(1 for case in medqa if key in case and case[key])
        percentage = (present / len(medqa)) * 100
        print(f"  {key}: {present}/{len(medqa)} ({percentage:.1f}%)")

    print(f"\n✓✓✓ DATASET READY FOR RESEARCH ✓✓✓")
    print("✓ This is the official USMLE medical exam dataset")
    print("✓ Ready for Notebook 00: Data Preparation")

US USMLE MEDQA DATASET ANALYSIS

✓ Total US USMLE cases: 12723

✓ Split distribution:
  dev: 1272 (10.0%)
  test: 1273 (10.0%)
  train: 10178 (80.0%)

✓ Specialty distribution:
  unknown: 12723 (100.0%)

✓ Sample questions by specialty:
  [unknown] A 23-year-old pregnant woman at 22 weeks gestation presents ...

✓ Data completeness check:
  question: 12723/12723 (100.0%)
  answer: 12723/12723 (100.0%)
  options: 12723/12723 (100.0%)
  specialty: 0/12723 (0.0%)

✓✓✓ DATASET READY FOR RESEARCH ✓✓✓
✓ This is the official USMLE medical exam dataset
✓ Ready for Notebook 00: Data Preparation


In [21]:
# SAVE DATASET TO GOOGLE DRIVE (FINAL STEP)
# This is the OUTPUT of Notebook 00
# Future notebooks (00B, 01, 02, etc) will load from here

import json
import os

print("=" * 70)
print("STEP 8: SAVE DATASET TO GOOGLE DRIVE")
print("=" * 70)

# The medqa_raw.json already exists in /content/data/
# Now we backup it to Google Drive for persistence

input_path = '/content/data/medqa_raw.json'

# Load it
with open(input_path, 'r', encoding='utf-8') as f:
    medqa = json.load(f)

print(f"\n✓ Dataset loaded from Colab storage")
print(f"  Total cases: {len(medqa):,}")

# Create output directory in Google Drive
output_dir = '/content/drive/My Drive/NS-MCA-Results'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'medqa_raw.json')

print(f"\nSaving to Google Drive...")
print(f"  Destination: {output_dir}/medqa_raw.json")

# Save to Google Drive
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(medqa, f, indent=2, ensure_ascii=False)

file_size_mb = os.path.getsize(output_path) / (1024*1024)

print(f"\n✓ Successfully saved to Google Drive!")
print(f"  File size: {file_size_mb:.1f} MB")
print(f"  Total cases: {len(medqa):,}")
print(f"  Location: /content/drive/My Drive/NS-MCA-Results/medqa_raw.json")

# Verify it's accessible
if os.path.exists(output_path):
    print(f"\n✓✓✓ VERIFICATION PASSED")
    print(f"✓ File exists at Google Drive destination")
    print(f"✓ Ready for downstream notebooks (00B, 01, 02...)")
else:
    print(f"\n✗ ERROR: File not saved correctly!")

print("\n" + "=" * 70)
print("✓✓✓ NOTEBOOK 00 COMPLETE ✓✓✓")
print("=" * 70)
print("\nOUTPUT SUMMARY:")
print(f"  Input:  data_clean.zip (125.6 MB)")
print(f"  Processing: Extract → Load → Validate")
print(f"  Output: medqa_raw.json ({file_size_mb:.1f} MB)")
print(f"  Destination: Google Drive (NS-MCA-Results folder)")
print(f"  Status: ✓ Ready for Notebook 00B (EDA)")

STEP 8: SAVE DATASET TO GOOGLE DRIVE

✓ Dataset loaded from Colab storage
  Total cases: 12,723

Saving to Google Drive...
  Destination: /content/drive/My Drive/NS-MCA-Results/medqa_raw.json

✓ Successfully saved to Google Drive!
  File size: 13.4 MB
  Total cases: 12,723
  Location: /content/drive/My Drive/NS-MCA-Results/medqa_raw.json

✓✓✓ VERIFICATION PASSED
✓ File exists at Google Drive destination
✓ Ready for downstream notebooks (00B, 01, 02...)

✓✓✓ NOTEBOOK 00 COMPLETE ✓✓✓

OUTPUT SUMMARY:
  Input:  data_clean.zip (125.6 MB)
  Processing: Extract → Load → Validate
  Output: medqa_raw.json (13.4 MB)
  Destination: Google Drive (NS-MCA-Results folder)
  Status: ✓ Ready for Notebook 00B (EDA)
